# 12 — Ax Bayesian optimization for damped_sw α (CTL-03)

**Notebook:** SOO — single-objective profit.

**Decision:** ADR 0060 / CTL-03=B — tune the demand fractile `alpha` for a ladder controller by maximizing closed-loop **episode profit** (SIM-01=B), using [Ax](https://ax.dev/) instead of a fixed grid.

Each Ax trial evaluates one candidate **(α, ρ)** pair on **K stochastic demand realizations** (distinct `root_seed`s). We report `(mean, sem)` to Ax so observation noise is explicit.

Scoring uses `evaluate_alpha_episode_outcomes(TUNE_ARM, ...)` in `sim/alpha_tune.py` — **Rust-first** when `blueberries_voi._core` is built (including calendar demand via typed `_core.DemandProfile`).

With calendar demand on, protection targets use **Monte Carlo** sums of heterogeneous daily NB draws μ(day+k) (CAL-B4 / ADR 0134), not flat μ=30.

**Defaults are smoke-sized.** Set `FULL_RUN = True` for longer episodes and more BO trials.

**Policy:** damped survival-weighted (`damped_sw`) ordering only — damped survival-weighted ordering only.

**Calendar demand + BO:** use `FULL_RUN = True` (`n_burn=n_score=28`, four weeks) so burn-in covers the MWF order cadence. Smoke `n_burn=2` makes Ax objectives extremely noisy and can look flat even when the Rust kernel is healthy.

## Setup

From the repo root:

```bash
uv sync --extra notebooks --extra viz --extra rust --extra data
uv run maturin develop --manifest-path crates/voi_py/Cargo.toml
uv run jupyter lab
```

First `uv sync` with `ax-platform` may take several minutes (PyTorch + BoTorch).

Add `--extra data` if you set `USE_ABDELLA = True` (Parquet shipments).

After pulling α/ρ tuning changes, rerun both commands and **restart the kernel**.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig
from tqdm.auto import tqdm

from blueberries_voi.backend import rust_available, rust_core
from blueberries_voi.sim.alpha_tune import (
    DEFAULT_CI_ALPHAS,
    DEFAULT_DESKTOP_ALPHAS,
    evaluate_alpha_episode_outcomes,
    tune_alpha_grid,
)
from blueberries_voi.sim.bakeoff_damped_sw import protection_demand_quantile
from blueberries_voi.model import ModelParams
from blueberries_voi.model.demand_profile import load_demand_profile
from blueberries_voi.sim.order_schedule import DEFAULT_ORDER_SCHEDULE
from blueberries_voi.sim.profit import ProfitCosts
from blueberries_voi.sim.shipments import default_shipments, smoke_cool_shipments

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "blueberries_voi").is_dir():
    REPO_ROOT = REPO_ROOT.parent

POLICY = "damped_sw"
TUNE_ARM = "sw"

FULL_RUN = True
ALPHA_BOUNDS = (0.1, 0.9999)

if FULL_RUN:
    N_BURN, N_SCORE = 28, 28
    K_BO_SEEDS = 6
    N_AX_TRIALS = 20
    K_VAL_SEEDS = 5
    GRID_ALPHAS = DEFAULT_DESKTOP_ALPHAS
else:
    N_BURN, N_SCORE = 2, 5
    K_BO_SEEDS = 4
    N_AX_TRIALS = 10
    K_VAL_SEEDS = 3
    GRID_ALPHAS = tuple(DEFAULT_CI_ALPHAS)

RNG = np.random.default_rng(20260817)
BO_SEEDS = [int(RNG.integers(0, 2**31 - 1)) for _ in range(K_BO_SEEDS)]
VAL_SEEDS = [int(RNG.integers(0, 2**31 - 1)) for _ in range(K_VAL_SEEDS)]

OUTPUT_JSON = REPO_ROOT / "outputs" / "damped_sw_alpha_bo.json"


def damped_sw_budget_kwargs() -> dict[str, object]:
    """No rollout budgets — damped_sw only."""
    return {}


def damped_sw_budget_kwargs() -> dict[str, object]:
    """No rollout budgets — damped_sw only."""
    return {}


rust_fn = getattr(rust_core, "evaluate_alpha_tune_outcomes_py", None) if rust_core else None
print(f"policy: {POLICY}")
print(f"Rust kernel: {rust_available() and rust_fn is not None}")
print(f"α bounds: {ALPHA_BOUNDS}")
print(f"episode: n_burn={N_BURN}, n_score={N_SCORE}")
print(f"BO seeds (K={K_BO_SEEDS}): {BO_SEEDS}")
print(f"validation seeds: {VAL_SEEDS}")
print(f"Ax trials: {N_AX_TRIALS}")

%matplotlib inline
plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.grid": True, "grid.alpha": 0.3})

## Constant parameters

Edit the scalars below before running the objective / BO cells. These feed `ProfitCosts`, `ModelParams`, shipments, and `lead_time` on the evaluation path.

### Calendar demand (`USE_CALENDAR_DEMAND`)

When **True**, `MODEL_PARAMS` carries the committed FreshNet DOW×week profile (`data/freshnet/demand_profile.json`). Both **realized demand** in the episode and the SW protection target `d_star = protection_demand_quantile(α, …, start_day=order_day)` use calendar μ(day).

- **Flat μ path** (`USE_CALENDAR_DEMAND = False`): legacy homogeneous NB over the protection window (length still 3/3/4 on Sun/Tue/Thu).
- **Calendar path**: heterogeneous MC sum (n_mc=20_000, ADR 0134) over the days in each protection window.

**Rust note:** when `_core` is built, `alpha_tune` stays **Rust-first** with calendar demand via typed `_core.DemandProfile` (T-133). Without `_core`, evaluation falls back to Python.


In [ ]:
# --- Episode economics (SIM-01=B; ADR 0104 scaffold — uncalibrated) ---
UNIT_MARGIN = 2.0
WASTE_COST = 5.0
STOCKOUT_PENALTY = 3.0

costs = ProfitCosts(
    unit_margin=UNIT_MARGIN,
    waste_cost=WASTE_COST,
    stockout_penalty=STOCKOUT_PENALTY,
)

# --- Shipments ---
USE_ABDELLA = False  # True → `default_shipments()`; needs `uv sync --extra data`

# --- Demand / spoilage / ordering ---
DEMAND_MU = 30.0
DEMAND_VM = 2.0  # variance/mean; must be > 1
CASE_SIZE = 8
LEAD_TIME = 1  # days (passed to Rust kernel when _core is available)

# CAL-B4: FreshNet calendar profile (ADR 0113 / T-132 MC protection quantile)
USE_CALENDAR_DEMAND = True
DEMAND_PROFILE_PATH = REPO_ROOT / "data" / "freshnet" / "demand_profile.json"

# SW damping (Nahmias); tuned jointly with alpha for damped_sw
RHO_BOUNDS = (0.5, 1.0)
DEFAULT_RHO = 0.8

_demand_profile = (
    load_demand_profile(DEMAND_PROFILE_PATH) if USE_CALENDAR_DEMAND else None
)
MODEL_PARAMS = ModelParams(
    demand_mu=DEMAND_MU,
    demand_vm=DEMAND_VM,
    case_size=CASE_SIZE,
    demand_profile=_demand_profile,
)

shipments = default_shipments() if USE_ABDELLA else smoke_cool_shipments()

_eval_backend = (
    "rust"
    if rust_available() and rust_fn is not None
    else "python"
)

print(f"costs: margin={UNIT_MARGIN}, waste={WASTE_COST}, stockout={STOCKOUT_PENALTY}")
print(f"shipments: {'Abdella' if USE_ABDELLA else 'smoke-cool'} ({len(shipments)} trace(s))")
print(f"model: μ={DEMAND_MU}, V/M={DEMAND_VM}, case={CASE_SIZE}, lead_time={LEAD_TIME}")
print(f"calendar demand: {USE_CALENDAR_DEMAND} → eval backend: {_eval_backend}")
print(f"α bounds: {ALPHA_BOUNDS}; ρ bounds: {RHO_BOUNDS}")


## Calendar protection targets (diagnostic)

Compare flat-μ vs calendar MC protection quantiles `d_star` on MWF order days. This is the quantity inside `q = case_round(ρ · max(0, d_star − Ĩ))`.


In [ ]:
ALPHA_DIAG = 0.9
flat_params = ModelParams(demand_mu=DEMAND_MU, demand_vm=DEMAND_VM)
schedule = DEFAULT_ORDER_SCHEDULE
order_days = [(d, schedule.protection_days(d)) for d in range(14) if schedule.can_order(d)]

rows = []
for day, prot in order_days[:3]:
    d_flat = protection_demand_quantile(
        ALPHA_DIAG, flat_params, protection_days=prot, start_day=day
    )
    d_cal = protection_demand_quantile(
        ALPHA_DIAG, MODEL_PARAMS, protection_days=prot, start_day=day
    )
    mus = [MODEL_PARAMS.demand_mu_for_day(day + k) for k in range(prot)]
    rows.append((day, prot, float(np.mean(mus)), d_flat, d_cal, d_cal - d_flat))

print(f"α={ALPHA_DIAG}: flat μ={DEMAND_MU} vs calendar profile")
for day, prot, mu_bar, d_flat, d_cal, delta in rows:
    print(
        f"  day={day:2d} prot={prot}  mean μ≈{mu_bar:5.1f}  "
        f"d*_flat={d_flat:6.1f}  d*_cal={d_cal:6.1f}  Δ={delta:+6.1f}"
    )


## Objective: episode outcomes with demand replicates

One Ax observation per (α, ρ) tuple = mean and SEM of K episode runs at fixed `BO_SEEDS`.


In [ ]:
def _episode_kwargs() -> dict[str, object]:
    return {
        "params": MODEL_PARAMS,
        "shipments": shipments,
        "costs": costs,
        "n_burn": N_BURN,
        "n_score": N_SCORE,
        "lead_time": LEAD_TIME,
        **damped_sw_budget_kwargs(),
    }


def evaluate_arm_outcomes(alpha: float, rho: float, root_seed: int):
    return evaluate_alpha_episode_outcomes(
        TUNE_ARM,
        float(alpha),
        int(root_seed),
        rho=float(rho),
        **_episode_kwargs(),
    )


def _replicate_mean_sem(values: list[float]) -> tuple[float, float]:
    arr = np.asarray(values, dtype=float)
    mean = float(arr.mean())
    sem = float(arr.std(ddof=1) / np.sqrt(len(arr))) if len(arr) > 1 else 0.0
    return mean, sem


def evaluate_with_replicates(
    alpha: float,
    rho: float,
    seeds: list[int],
) -> dict[str, tuple[float, float]]:
    """Mean and SEM over K demand seeds for profit / waste / stockout metrics."""
    profits: list[float] = []
    wastes: list[float] = []
    stockouts: list[float] = []
    for seed in seeds:
        out = evaluate_arm_outcomes(alpha, rho, seed)
        profits.append(out.profit)
        wastes.append(float(out.total_waste))
        stockouts.append(float(out.total_lost_sales))
    p_mean, p_sem = _replicate_mean_sem(profits)
    w_mean, w_sem = _replicate_mean_sem(wastes)
    s_mean, s_sem = _replicate_mean_sem(stockouts)
    return {
        "episode_profit": (p_mean, p_sem),
        "total_waste": (w_mean, w_sem),
        "total_stockout": (s_mean, s_sem),
    }


def ax_parameter_configs() -> list[RangeParameterConfig]:
    return [
        RangeParameterConfig(
            name="alpha",
            parameter_type="float",
            bounds=ALPHA_BOUNDS,
        ),
        RangeParameterConfig(
            name="rho",
            parameter_type="float",
            bounds=RHO_BOUNDS,
        ),
    ]


demo = evaluate_with_replicates(0.9, DEFAULT_RHO, BO_SEEDS[:2])
print(
    f"smoke {TUNE_ARM} α=0.9 ρ={DEFAULT_RHO} on 2 seeds: "
    f"profit={demo['episode_profit'][0]:.2f}±{demo['episode_profit'][1]:.3f}, "
    f"waste={demo['total_waste'][0]:.1f}, stockout={demo['total_stockout'][0]:.1f}"
)

def plot_bo_run(
    client: Client,
    trial_log: list[dict[str, Any]],
    *,
    title: str,
    best_params: dict[str, float],
    objective_key: str = "mean_profit",
) -> None:
    """Matplotlib summary for a 2D (alpha, rho) BO run."""
    alphas = np.array([t["alpha"] for t in trial_log])
    rhos = np.array([t["rho"] for t in trial_log])
    profits = np.array([t[objective_key] for t in trial_log])
    sems = np.array([t["sem_profit"] for t in trial_log])
    trials = np.array([t["trial_index"] for t in trial_log])
    best_so_far = np.maximum.accumulate(profits)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

    sc = axes[0].scatter(alphas, rhos, c=profits, cmap="viridis", s=55, alpha=0.85)
    axes[0].scatter(
        [best_params["alpha"]],
        [best_params["rho"]],
        marker="*",
        s=220,
        c="#16a34a",
        label="best",
        zorder=5,
    )
    axes[0].set_xlabel("α")
    axes[0].set_ylabel("ρ")
    axes[0].set_title(f"{title}: trial locations")
    fig.colorbar(sc, ax=axes[0], label="replicate mean profit")
    axes[0].legend(fontsize=8)

    axes[1].errorbar(alphas, profits, yerr=sems, fmt="o", alpha=0.75, capsize=3)
    axes[1].set_xlabel("α")
    axes[1].set_ylabel("Episode profit (replicate mean)")
    axes[1].set_title("Profit vs α")

    axes[2].plot(trials, profits, "o", alpha=0.5, label="trial mean")
    axes[2].plot(trials, best_so_far, "-", color="#16a34a", lw=2, label="best-so-far")
    axes[2].set_xlabel("Ax trial index")
    axes[2].set_ylabel("Replicate mean profit")
    axes[2].set_title("Convergence")
    axes[2].legend(fontsize=8)

    fig.suptitle(title, y=1.02)
    fig.tight_layout()
    plt.show()


## Textbook fractile reference (CTL-03 context)

In [ ]:
alpha_theory_penalty = costs.stockout_penalty / (
    costs.stockout_penalty + costs.waste_cost
)
alpha_theory_margin = costs.unit_margin / (costs.unit_margin + costs.waste_cost)
print(f"Textbook (penalty / waste): {alpha_theory_penalty:.3f}")
print(f"Textbook (margin / waste):  {alpha_theory_margin:.3f}")

## Run 1 — single-objective profit maximization

Joint Ax search over **(α, ρ)** maximizing `episode_profit` ([Ax getting started](https://ax.dev/docs/tutorials/getting_started/)).

In [ ]:
client_profit = Client()
client_profit.configure_experiment(parameters=ax_parameter_configs())
client_profit.configure_optimization(objective="episode_profit")

trial_log_profit: list[dict[str, Any]] = []
for _ in tqdm(range(N_AX_TRIALS), desc="Ax profit SOO"):
    trials = client_profit.get_next_trials(max_trials=1)
    trial_index, parameters = next(iter(trials.items()))
    alpha = float(parameters["alpha"])
    rho = float(parameters["rho"])
    metrics = evaluate_with_replicates(alpha, rho, BO_SEEDS)
    client_profit.complete_trial(
        trial_index=trial_index,
        raw_data={"episode_profit": metrics["episode_profit"]},
    )
    trial_log_profit.append(
        {
            "trial_index": int(trial_index),
            "alpha": alpha,
            "rho": rho,
            "mean_profit": metrics["episode_profit"][0],
            "sem_profit": metrics["episode_profit"][1],
            "mean_waste": metrics["total_waste"][0],
            "mean_stockout": metrics["total_stockout"][0],
        }
    )

best_profit_params, _pred, best_profit_index, _name = client_profit.get_best_parameterization()
best_alpha_profit = float(best_profit_params["alpha"])
best_rho_profit = float(best_profit_params["rho"])
print(
    f"SOO best (model): α={best_alpha_profit:.4f}, ρ={best_rho_profit:.4f} "
    f"(trial {best_profit_index})"
)


## Run 1 diagnostics (profit SOO)

In [ ]:
from ax.analysis.plotly import SlicePlot

if len(trial_log_profit) >= 8:
    client_profit.compute_analyses(analyses=[SlicePlot(parameter_name="alpha")])
    client_profit.compute_analyses(analyses=[SlicePlot(parameter_name="rho")])

plot_bo_run(
    client_profit,
    trial_log_profit,
    title=f"Run 1 — profit SOO ({TUNE_ARM})",
    best_params={"alpha": best_alpha_profit, "rho": best_rho_profit},
)


## Grid baseline on the same BO seed panel

For each grid α, use the same replicate-mean objective (not `tune_alpha_grid`'s single-seed CRN).

In [ ]:
GRID_RHO = best_rho_profit
grid_means: list[float] = []
grid_sems: list[float] = []
for a in tqdm(GRID_ALPHAS, desc="grid baseline"):
    profit_stats = evaluate_with_replicates(float(a), GRID_RHO, BO_SEEDS)["episode_profit"]
    grid_means.append(profit_stats[0])
    grid_sems.append(profit_stats[1])

best_grid_idx = int(np.argmax(grid_means))
best_alpha_grid = float(GRID_ALPHAS[best_grid_idx])
print(f"Grid best α at ρ={GRID_RHO:.3f}: {best_alpha_grid:.4f}")

crn_seed = BO_SEEDS[0]
best_alpha_crn = tune_alpha_grid(
    TUNE_ARM,
    alphas=GRID_ALPHAS,
    root_seed=crn_seed,
    params=MODEL_PARAMS,
    shipments=shipments,
    costs=costs,
    n_burn=N_BURN,
    n_score=N_SCORE,
    **damped_sw_budget_kwargs(),
)
print(f"tune_alpha_grid({TUNE_ARM}) on seed {crn_seed}: α*={best_alpha_crn:.4f}")


In [ ]:
def validation_mean(alpha: float, rho: float) -> float:
    return float(
        np.mean([evaluate_arm_outcomes(alpha, rho, s).profit for s in VAL_SEEDS])
    )


val_profit = validation_mean(best_alpha_profit, best_rho_profit)
val_grid = validation_mean(best_alpha_grid, GRID_RHO)
print(
    f"Validation profit — SOO α={best_alpha_profit:.4f} ρ={best_rho_profit:.4f}: {val_profit:.2f}"
)
print(f"Validation profit — grid α={best_alpha_grid:.4f} ρ={GRID_RHO:.4f}: {val_grid:.2f}")
    print(
    )


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.errorbar(
    [t["alpha"] for t in trial_log_profit],
    [t["mean_profit"] for t in trial_log_profit],
    yerr=[t["sem_profit"] for t in trial_log_profit],
    fmt="o",
    alpha=0.7,
    label="SOO trials",
)
ax.plot(GRID_ALPHAS, grid_means, "s--", color="#2563eb", label="grid baseline")
ax.axvline(best_alpha_profit, color="#16a34a", ls="--", label=f"SOO α*={best_alpha_profit:.2f}")
ax.axvline(best_alpha_grid, color="#9333ea", ls=":", label=f"grid α*={best_alpha_grid:.2f}")
ax.axvline(alpha_theory_penalty, color="#dc2626", ls=":", alpha=0.7, label="theory (penalty)")
ax.set_xlabel("α")
ax.set_ylabel("Episode profit (replicate mean)")
ax.set_title(f"SOO vs grid — {TUNE_ARM} (ρ={GRID_RHO:.2f})")
ax.legend(fontsize=8, loc="best")
fig.tight_layout()
plt.show()


## Save results (optional)

Writes to `outputs/` (gitignored) — not `experiments/tuned_alpha.json`.

In [ ]:
payload: dict[str, Any] = {
    "policy": POLICY,
    "full_run": FULL_RUN,
    "rust_kernel": bool(rust_available() and rust_fn is not None),
    "alpha_bounds": list(ALPHA_BOUNDS),
    "rho_bounds": list(RHO_BOUNDS),
    "n_burn": N_BURN,
    "n_score": N_SCORE,
    "bo_seeds": BO_SEEDS,
    "val_seeds": VAL_SEEDS,
    "costs": {
        "unit_margin": UNIT_MARGIN,
        "waste_cost": WASTE_COST,
        "stockout_penalty": STOCKOUT_PENALTY,
    },
    "model_params": {
        "demand_mu": DEMAND_MU,
        "demand_vm": DEMAND_VM,
        "use_calendar_demand": USE_CALENDAR_DEMAND,
        "case_size": CASE_SIZE,
        "lead_time": LEAD_TIME,
        "use_abdella": USE_ABDELLA,
    },
    "best_alpha_profit_soo": best_alpha_profit,
    "best_rho_profit_soo": best_rho_profit,
    "best_alpha_grid": best_alpha_grid,
    "best_alpha_tune_alpha_grid_crn": float(best_alpha_crn),
    "validation_mean_profit_soo": val_profit,
    "validation_mean_grid": val_grid,
    "trials_profit_soo": trial_log_profit,
}
OUTPUT_JSON.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_JSON.write_text(json.dumps(payload, indent=2) + "\n", encoding="utf-8")
print(f"Wrote {OUTPUT_JSON}")


## Takeaways

1. **damped_sw** = survival-weighted base-stock: `q = case_round(ρ · max(0, F⁻¹(α) − Ĩ))`.
2. Ax receives `(mean, sem)` per metric over K demand seeds.
3. Rebuild `maturin develop` after pulling ρ-aware `voi_core` changes.
4. With calendar demand, use `FULL_RUN = True` so burn-in covers MWF cadence.
